# 01-data-mesh-delfos.py

01-data-mesh-delfos.py — Modulo 1: Introduccion a Data Mesh aplicado a Delfos (45 min)

In [0]:
import re, json
from datetime import datetime
from pyspark.sql import functions as F, DataFrame
from pyspark.sql.window import Window
from typing import List

for _n, _d, _l in [
    ("s3_bucket", "",           "S3 Bucket (vacio = secrets)"),
    ("catalog",   "nequi_prod", "Catalogo Unity Catalog (debe existir)"),
    ("dominio",   "pagos",      "Dominio de negocio"),
    ("nickname",  "",           "Nickname / iniciales (sufijo de schemas, ej: jdoe)"),
]:
    try:    dbutils.widgets.get(_n)
    except: dbutils.widgets.text(_n, _d, _l)

try: dbutils.widgets.get("reset")
except: dbutils.widgets.dropdown("reset","No",["No","Si — reiniciar datos"],"Reiniciar")

CATALOG = dbutils.widgets.get("catalog").strip() or "nequi_prod"
DOMINIO = dbutils.widgets.get("dominio").strip() or "pagos"
RESET   = dbutils.widgets.get("reset").startswith("Si")
_nick_raw = dbutils.widgets.get("nickname").strip().lower()
if not _nick_raw:
    _nick_raw = spark.sql("SELECT current_user()").collect()[0][0].split("@")[0]
NICK = re.sub(r"[^a-z0-9]", "", _nick_raw)[:15]
assert NICK, "No se pudo determinar el nickname — llena el widget 'nickname' con tus iniciales (ej: jdoe)"

# Refrescar widgets con los valores computados para que las celdas %sql resuelvan
# ${catalog} y ${nickname} correctamente (las celdas SQL leen el widget, no la variable Python)
for _wn, _wv, _wl in [
    ("catalog",  CATALOG, "Catalogo Unity Catalog (debe existir)"),
    ("nickname", NICK,    "Nickname / iniciales (sufijo de schemas, ej: jdoe)"),
]:
    try: dbutils.widgets.remove(_wn)
    except: pass
    dbutils.widgets.text(_wn, _wv, _wl)

In [0]:
%run ./_resource/00-setup

# Modulo 1 — Introduccion a Data Mesh aplicado a Delfos
**45 minutos** &nbsp;|&nbsp; Dominios de negocio · Data Products · Gobernanza federada · Plataforma self-serve

En este modulo construimos el primer Data Product de Delfos desde cero: desde la estructura
del catalogo hasta el contrato de gobernanza, la seguridad a nivel de columna y la
trazabilidad regulatoria que exige la SFC.

## Por que Data Mesh en Nequi?

Nequi tiene un problema clasico de escala: el equipo central de datos se convirtio en un cuello
de botella. Pagos necesita un dashboard, riesgo necesita features para el modelo, producto
necesita una cohorte — los tres hacen fila con el mismo equipo de ingenieria de datos,
que no puede atender todo a la vez.

**Data Mesh** (Zhamak Dehghani, 2019) propone un cambio de paradigma: en lugar de centralizar
el dato, **descentralizar la responsabilidad** de producirlo hacia los equipos que mejor lo
conocen. Delfos es la implementacion de esos principios en Nequi.

## Los 4 principios de Data Mesh — implementacion en Delfos

| Principio | Descripcion | Implementacion en Delfos |
|---|---|---|
| **1. Propiedad orientada al dominio** | Cada equipo es dueno de sus datos. Nadie mas puede publicar en su schema. | `{catalog}.pagos_{nick}.*` · `{catalog}.riesgo_{nick}.*` · `{catalog}.clientes_{nick}.*` |
| **2. Datos como producto** | El Data Product incluye propietario, SLA, schema documentado, tests de calidad y clasificacion de seguridad. | Unity Catalog + `TBLPROPERTIES` + Great Expectations |
| **3. Plataforma self-serve** | Los equipos crean pipelines y consultan datos sin depender del equipo central. | Building Blocks reutilizables + Asset Bundles + Unity Catalog autodescubrible |
| **4. Gobernanza federada** | Autonomia por dominio respetando estandares globales: SARLAFT, Circular 052 SFC, Habeas Data. | Unity Catalog RLS + Column Masks + Audit Logs en `system.access.audit` |

### 1.1 — Estructura de dominios en Unity Catalog

El setup de Delfos crea cuatro schemas en el catalogo configurado en el widget, uno por dominio de negocio.
Cada schema es propiedad exclusiva del equipo correspondiente.
Observa que la estructura es plana: `catalogo.dominio.tabla` — sin jerarquias adicionales.

### 1.1 — Dominios registrados en Unity Catalog

In [0]:
# El catalogo organiza los datos por dominio de negocio.
# Esta es la estructura de gobierno que habilita el Principio 1 de Data Mesh.
spark.sql(f"SHOW SCHEMAS IN {CATALOG}").display()

### 1.2 — Cargar datos de muestra para el modulo

Para poder demostrar todos los conceptos de Data Mesh en este modulo sin depender del
pipeline completo de S3 (Modulo 2), generamos 2.500 transacciones sinteticas directamente
en Spark e inyectamos patrones de fraude reales para que las demostraciones de gobernanza
sean significativas.

Los datos siguen el schema exacto de produccion — incluyendo distribucion realista de canales,
ciudades colombianas y patrones de monto log-normales (media ~85.000 COP, tipico de Nequi).

### 1.2 — Generacion de datos sinteticos para el modulo

In [0]:
import pandas as pd, uuid, random
from datetime import datetime, timedelta

random.seed(42)

CANALES  = ["app", "qr", "corresponsal", "api"]
CIUDADES = ["Bogota", "Medellin", "Cali", "Barranquilla", "Cartagena",
            "Bucaramanga", "Pereira", "Cucuta", "Ibague", "Santa Marta"]
N_USERS, N_TX = 80, 2_500
users = [f"u_{i:04d}" for i in range(1, N_USERS + 1)]
now   = datetime.now()

# Marcar 8% de usuarios como potencialmente fraudulentos
fraud_monto = set(random.sample(users, 6))
fraud_freq  = set(random.sample(users, 5))

rows = []
for _ in range(N_TX):
    uid  = random.choice(users)
    ts   = now - timedelta(hours=random.randint(0, 168))
    monto_base = round(max(1_000, min(random.lognormvariate(11.35, 1.1), 9_999_999)), 2)
    es_fraude  = False

    # Inyectar fraude por monto (z-score alto)
    if uid in fraud_monto and random.random() < 0.05:
        monto_base = round(random.uniform(4_500_000, 9_800_000), 2)
        es_fraude  = True

    rows.append({
        "transaction_id": str(uuid.uuid4()),
        "user_id"       : uid,
        "monto"         : monto_base,
        "canal"         : random.choices(CANALES, weights=[55, 25, 12, 8])[0],
        "ciudad"        : random.choices(CIUDADES, weights=[30,22,13,9,7,5,3,3,4,4])[0],
        "dispositivo"   : f"{'Android' if random.random()<0.72 else 'iOS'}-{uid[-4:].upper()}",
        "ts"            : ts.strftime("%Y-%m-%dT%H:%M:%S"),
        "capa"          : "silver",          # cargamos directamente como silver (ya validado)
        "es_fraude_real": es_fraude,
        "_procesado_en" : now,
    })

# Agregar rafaga de frecuencia para usuarios marcados (patron de fraude por API)
for uid in list(fraud_freq)[:3]:
    ts_burst = now - timedelta(hours=random.randint(1, 48))
    for i in range(8):                        # 8 tx en 4 minutos: supera max_tx=5
        rows.append({
            "transaction_id": str(uuid.uuid4()),
            "user_id"       : uid,
            "monto"         : round(random.uniform(10_000, 80_000), 2),
            "canal"         : "api",
            "ciudad"        : "Bogota",
            "dispositivo"   : f"Android-{uid[-4:].upper()}",
            "ts"            : (ts_burst + timedelta(seconds=i*30)).strftime("%Y-%m-%dT%H:%M:%S"),
            "capa"          : "silver",
            "es_fraude_real": True,
            "_procesado_en" : now,
        })

df_muestra = spark.createDataFrame(pd.DataFrame(rows))
(df_muestra.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("capa")
    .saveAsTable(f"{CATALOG}.{SCH_PAGOS}.transacciones"))

n_total  = df_muestra.count()
n_fraude = df_muestra.filter("es_fraude_real").count()
print(f"Datos cargados en {CATALOG}.{SCH_PAGOS}.transacciones")
print(f"  Total       : {n_total:,} transacciones")
print(f"  Fraudulentas: {n_fraude:,}  ({100*n_fraude/n_total:.1f}%)")
print(f"  Usuarios    : {N_USERS}")
print(f"  Periodo     : ultimas 7 dias")

### 1.3 — Explorar el Data Product: perfil de los datos

Antes de registrar el contrato formal del Data Product, perfilamos los datos para entender
su distribucion. Esto es lo que el equipo de pagos haria antes de publicar la primera version:
verificar que los datos son los esperados y que el schema es correcto.

### 1.3a — Distribucion por canal de pago

In [0]:
# Distribucion de transacciones por canal: verificar que los pesos son realistas
# La app debe ser el canal dominante (~55% en Nequi)
spark.sql(f"""
SELECT
    canal,
    COUNT(*)                                         AS total_transacciones,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS porcentaje,
    ROUND(AVG(monto), 0)                             AS monto_promedio_cop,
    ROUND(MIN(monto), 0)                             AS monto_minimo,
    ROUND(MAX(monto), 0)                             AS monto_maximo,
    ROUND(PERCENTILE(monto, 0.5), 0)                 AS mediana_cop,
    ROUND(PERCENTILE(monto, 0.95), 0)                AS percentil_95
FROM {CATALOG}.{SCH_PAGOS}.transacciones
WHERE capa = 'silver'
GROUP BY canal
ORDER BY total_transacciones DESC
""").display()

### 1.3b — Distribucion geografica de transacciones

In [0]:
# Distribucion por ciudad: Bogota y Medellin concentran el grueso del volumen
spark.sql(f"""
SELECT
    ciudad,
    COUNT(*)                                           AS total,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS porcentaje,
    ROUND(AVG(monto), 0)                               AS monto_promedio_cop,
    SUM(monto) / 1e6                                   AS volumen_millones_cop
FROM {CATALOG}.{SCH_PAGOS}.transacciones
WHERE capa = 'silver'
GROUP BY ciudad
ORDER BY total DESC
""").display()

### 1.3c — Actividad por hora del dia (patron de uso)

In [0]:
# Patron de uso por hora: permite validar que los timestamps son coherentes con
# el comportamiento esperado de usuarios (picos en manana y tarde)
spark.sql(f"""
SELECT
    HOUR(ts)          AS hora_utc,
    COUNT(*)          AS transacciones,
    ROUND(AVG(monto)) AS monto_promedio
FROM {CATALOG}.{SCH_PAGOS}.transacciones
WHERE capa = 'silver'
GROUP BY hora_utc
ORDER BY hora_utc
""").display()

### 1.4 — Registrar el contrato del Data Product

Un **Data Product** en Delfos no es solo una tabla — es un contrato.
`TBLPROPERTIES` es el mecanismo de Unity Catalog para registrar ese contrato junto con el dato:
quien lo posee, con que SLA, bajo que normas regulatorias y en que estado del ciclo de vida.

El contrato vive con el dato — no en un documento externo que nadie actualiza.
Cualquier equipo puede consultar el contrato con `DESCRIBE EXTENDED` sin buscar documentacion adicional.

### 1.4a — Registrar el contrato completo con TBLPROPERTIES

In [0]:
spark.sql(f"""
ALTER TABLE {CATALOG}.{SCH_PAGOS}.transacciones
SET TBLPROPERTIES (
    'delfos.product_id'       = 'pagos.transacciones.v1',
    'delfos.domain'           = 'pagos',
    'delfos.owner_team'       = 'equipo-pagos',
    'delfos.owner_email'      = 'datos-pagos@nequi.com.co',
    'delfos.sla_freshness'    = '1h',
    'delfos.sla_availability' = '99.5%',
    'delfos.quality_tests'    = 'great_expectations::pagos_suite_v1',
    'delfos.classification'   = 'CONFIDENCIAL',
    'delfos.pii_columns'      = 'user_id,dispositivo',
    'delfos.regulatory'       = 'SARLAFT,Circular-052-SFC,Habeas-Data-1581',
    'delfos.retention_days'   = '1825',
    'delfos.version'          = '1.0.0',
    'delfos.status'           = 'PRODUCCION'
)
""")
spark.sql(f"""
COMMENT ON TABLE {CATALOG}.{SCH_PAGOS}.transacciones IS
    'Data Product: transacciones Nequi normalizadas y validadas. Fuente de verdad del dominio pagos. SLA: 1h. SARLAFT.'
""")
print(f"Contrato registrado en {CATALOG}.{SCH_PAGOS}.transacciones")

### 1.4b — Verificar el contrato registrado

In [0]:
# DESCRIBE EXTENDED muestra TBLPROPERTIES junto con estadisticas fisicas de la tabla.
# Un analista de cumplimiento puede ejecutar esto para auditar el contrato sin acceder al codigo.
spark.sql(f"DESCRIBE EXTENDED {CATALOG}.{SCH_PAGOS}.transacciones").display()

### 1.5 — Documentar columnas del Data Product

El contrato de la tabla no es suficiente. Cada columna debe estar documentada con:
su significado de negocio, unidad, valores validos y si contiene PII.

Unity Catalog almacena estos comentarios y los expone en el catalogo de datos,
en los resultados de `DESCRIBE TABLE` y en la UI de Databricks — no en una wiki externa
que siempre esta desactualizada.

### 1.5 — Documentar columnas con ALTER COLUMN COMMENT

In [0]:
_t = f"{CATALOG}.{SCH_PAGOS}.transacciones"
for _col, _comment in [
    ("transaction_id", "UUID unico de la transaccion. Clave primaria del Data Product."),
    ("user_id",        "Identificador del usuario Nequi. PII — enmascarar en ambientes no-prod."),
    ("monto",          "Monto de la transaccion en pesos colombianos (COP). Siempre positivo."),
    ("canal",          "Canal de origen: app | qr | corresponsal | api. Validado en ingesta."),
    ("ciudad",         "Ciudad de origen de la transaccion segun geolocalizacion del dispositivo."),
    ("dispositivo",    "Hash del dispositivo movil. PII — enmascarar en ambientes no-prod."),
    ("ts",             "Timestamp de la transaccion en UTC (ISO 8601). Nunca en el futuro."),
    ("capa",           "Capa Medallion donde reside el registro: bronze | silver | gold."),
]:
    spark.sql(f"ALTER TABLE {_t} ALTER COLUMN {_col} COMMENT '{_comment}'")
    print(f"  {_col}: OK")

### 1.5 (verificacion) — Confirmar comentarios registrados

In [0]:
spark.sql(f"DESCRIBE TABLE {CATALOG}.{SCH_PAGOS}.transacciones").display()

### 1.6 — DESCRIBE HISTORY: el log de transacciones de Delta

Delta Lake mantiene un **log de transacciones** inmutable que registra cada operacion
sobre la tabla: quien escribio, cuantos registros, cuando y que operacion fue.

Este log es el fundamento de la auditoria regulatoria en Delfos: la SFC puede solicitar
"muestra el estado de los datos de pagos el 15 de marzo a las 10 AM" y Delfos puede
responder en segundos con Delta Time Travel — sin necesidad de un sistema separado de auditoria.

### 1.6 — Log de transacciones Delta (historial completo de la tabla)

In [0]:
# Cada fila es una version de la tabla.
# 'operation' muestra el tipo de escritura: WRITE, MERGE, DELETE, ALTER TABLE...
spark.sql(f"DESCRIBE HISTORY {CATALOG}.{SCH_PAGOS}.transacciones").display()

### 1.7 — Gobernanza federada: Row-Level Security y Column Mask

El **Principio 4** (gobernanza federada) en accion: el dominio de riesgo define quien puede
ver que dentro de sus propios datos, respetando los estandares globales de Nequi.

La vista `v_transacciones` aplica dos controles simultaneos:
- **Row-Level Security**: el equipo de riesgo solo ve registros propios de su dominio.
- **Column Mask**: la columna `dispositivo` (PII) se enmascara para roles sin privilegio.

Ambos controles son **transparentes para el consumidor**: consulta la vista como cualquier
tabla SQL y Unity Catalog aplica los filtros automaticamente segun el usuario activo.

### 1.7a — Crear vista con RLS y Column Mask

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{SCH_RIESGO}.v_transacciones
COMMENT 'Vista del dominio riesgo sobre transacciones de pagos. RLS + Column Mask activos.'
AS
SELECT
    transaction_id,
    user_id,
    monto,
    canal,
    ciudad,
    ts,
    CASE WHEN is_member('equipo-cumplimiento') OR is_member('admins')
         THEN dispositivo ELSE '***' END AS dispositivo,
    capa,
    es_fraude_real,
    _procesado_en
FROM {CATALOG}.{SCH_PAGOS}.transacciones
WHERE capa = 'silver'
   OR is_member('admins')
   OR is_member('equipo-cumplimiento')
""")
print(f"Vista creada: {CATALOG}.{SCH_RIESGO}.v_transacciones")

### 1.7b — Verificar que el enmascaramiento funciona para el usuario actual

In [0]:
# El usuario del taller NO pertenece a equipo-cumplimiento ni admins,
# por lo tanto debe ver '***' en la columna dispositivo.
spark.sql(f"""
SELECT
    current_user()              AS yo,
    COUNT(*)                    AS registros_visibles,
    COUNT(DISTINCT dispositivo) AS dispositivos_distintos,
    FIRST(dispositivo)          AS muestra_dispositivo
FROM {CATALOG}.{SCH_RIESGO}.v_transacciones
""").display()

### 1.8 — Delta Time Travel: auditoria regulatoria

Una de las capacidades mas importantes de Delta Lake para el cumplimiento regulatorio.
La SFC puede pedir: *"muestra el estado de los datos de pagos al momento de la auditoria"*.
Con Delta Time Travel, Delfos puede reconstruir el estado exacto de cualquier tabla
en cualquier version o timestamp historico — sin sistemas de auditoria externos.

Para que el demo sea significativo primero simulamos una segunda escritura
(como si el pipeline hubiera corrido una segunda vez con nuevas transacciones).

### 1.8a — Simular una segunda carga para crear la version 1 de la tabla

In [0]:
import pandas as pd, uuid, random
from datetime import datetime, timedelta

random.seed(99)
now = datetime.now()

# Generar 200 transacciones adicionales — simula el pipeline del dia siguiente
filas_v2 = []
for _ in range(200):
    uid = f"u_{random.randint(1, 80):04d}"
    filas_v2.append({
        "transaction_id": str(uuid.uuid4()),
        "user_id"       : uid,
        "monto"         : round(max(1_000, random.lognormvariate(11.5, 0.9)), 2),
        "canal"         : random.choices(["app","qr","corresponsal","api"], weights=[55,25,12,8])[0],
        "ciudad"        : random.choice(["Bogota","Medellin","Cali","Barranquilla"]),
        "dispositivo"   : f"Android-{uid[-4:].upper()}",
        "ts"            : (now - timedelta(hours=random.randint(0, 24))).strftime("%Y-%m-%dT%H:%M:%S"),
        "capa"          : "silver",
        "es_fraude_real": False,
        "_procesado_en" : now,
    })

df_v2 = spark.createDataFrame(pd.DataFrame(filas_v2))
(df_v2.write.format("delta").mode("append")
    .partitionBy("capa")
    .saveAsTable(f"{CATALOG}.{SCH_PAGOS}.transacciones"))

print(f"Version 1 creada: 200 registros adicionales agregados con mode='append'")
print(f"La tabla ahora tiene 2 versiones en el log de Delta.")

### 1.8b — DESCRIBE HISTORY: ver las dos versiones disponibles

In [0]:
# Ahora hay dos versiones: la carga inicial (version 0) y la segunda carga (version 1).
# Cada fila del log es una operacion de escritura — inmutable e irrepetible.
spark.sql(f"DESCRIBE HISTORY {CATALOG}.{SCH_PAGOS}.transacciones").display()

### 1.8c — Time Travel: comparar version 0 vs version actual

In [0]:
# La version 0 tiene 2.500 registros (carga inicial del modulo).
# La version actual tiene 2.700 registros (2.500 + 200 de la segunda carga).
# Esto es lo que un auditor de la SFC consulta para reconstruir el estado historico.
spark.sql(f"""
SELECT 'Version 0 (carga inicial)'   AS momento, COUNT(*) AS total, ROUND(AVG(monto)) AS monto_prom
FROM {CATALOG}.{SCH_PAGOS}.transacciones VERSION AS OF 0
UNION ALL
SELECT 'Version 1 (carga siguiente)' AS momento, COUNT(*) AS total, ROUND(AVG(monto)) AS monto_prom
FROM {CATALOG}.{SCH_PAGOS}.transacciones VERSION AS OF 1
UNION ALL
SELECT 'Version actual'              AS momento, COUNT(*) AS total, ROUND(AVG(monto)) AS monto_prom
FROM {CATALOG}.{SCH_PAGOS}.transacciones
""").display()

### 1.8d — Time Travel: auditar una transaccion especifica en la version 0

In [0]:
# Escenario real de auditoria SFC: verificar el monto original de una transaccion
# antes de cualquier transformacion posterior. VERSION AS OF 0 garantiza
# que vemos el dato tal como fue cargado inicialmente.
spark.sql(f"""
SELECT
    transaction_id,
    user_id,
    monto,
    canal,
    ciudad,
    ts,
    _procesado_en
FROM {CATALOG}.{SCH_PAGOS}.transacciones VERSION AS OF 0
ORDER BY _procesado_en ASC
LIMIT 5
""").display()

### 1.9 — Descubrimiento self-serve: catalogo de Data Products

**Principio 3** (plataforma self-serve): un analista del equipo de clientes puede descubrir
que Data Products existen en Delfos sin necesitar al equipo de ingenieria.

La vista `{catalog}.information_schema.tables` expone todos los objetos del catalogo con
sus metadatos. Esta es la primera consulta que corre cualquier equipo nuevo que quiere
saber que datos tiene disponibles para consumir.

### 1.9a — Catalogo de Data Products (schemas del participante)

In [0]:
spark.sql(f"""
SELECT
    table_schema   AS dominio,
    table_name     AS data_product,
    table_type,
    comment        AS descripcion,
    created        AS creado_en,
    last_altered   AS ultima_modificacion
FROM {CATALOG}.information_schema.tables
WHERE table_schema NOT IN ('information_schema')
ORDER BY dominio, data_product
""").display()

### 1.9b — Consultar el contrato de un Data Product especifico

In [0]:
# SHOW TBLPROPERTIES expone el contrato completo del Data Product.
# Un consumidor puede ejecutar esto sin buscar documentacion en otra herramienta.
spark.sql(f"SHOW TBLPROPERTIES {CATALOG}.{SCH_PAGOS}.transacciones").display()

### 1.10 — Auditoria de acceso: system.access.audit

Unity Catalog registra automaticamente cada acceso a los Data Products en `system.access.audit`.
Esta tabla es exactamente lo que la SFC solicita en una auditoria SARLAFT: quien accedio
a que dato y cuando — sin instrumentacion adicional de parte del equipo de datos.

> **Nota:** `system.access.audit` requiere Unity Catalog Enterprise o Premium con
> System Tables habilitadas. En cuentas trial puede no estar disponible.
> Para habilitarlo: Account Console > Unity Catalog > Metastore > System Tables.

### 1.10 — Registro de accesos a Data Products de Delfos

In [0]:
# En Unity Catalog, request_params es MAP<STRING,STRING> — acceso con corchetes, no punto.
# Envolvemos en try/except porque system.access.audit requiere permisos especiales.
try:
    spark.sql(f"""
        SELECT
            DATE_TRUNC('minute', event_time)   AS minuto,
            user_identity.email                AS usuario,
            action_name,
            request_params['table_full_name']  AS data_product_accedido
        FROM system.access.audit
        WHERE service_name = 'unityCatalog'
          AND action_name IN ('getTable','selectFromTable','createTable','alterTable')
          AND request_params['table_full_name'] LIKE '{CATALOG}%'
          AND event_time >= current_timestamp() - INTERVAL 3 HOURS
        ORDER BY event_time DESC
        LIMIT 20
    """).display()
except Exception as _e:
    _msg = str(_e).lower()
    if any(k in _msg for k in ["access_denied", "permission", "not found", "not authorized"]):
        print("[ATENCION] system.access.audit no esta disponible en esta cuenta.")
        print("  Requiere Unity Catalog Enterprise/Premium con System Tables habilitadas.")
        print("  Para habilitar: Account Console > Unity Catalog > Metastore > System Tables > Enable")
        print()
        print("  Lo que registraria en produccion (ultimas 3 horas):")
        print(f"  {'minuto':<20} {'usuario':<30} {'action_name':<20} data_product_accedido")
        print(f"  {'-'*20} {'-'*30} {'-'*20} {'-'*40}")
        print(f"  2024-01-15 11:02     analista@nequi.com.co          getTable             {CATALOG}.{SCH_PAGOS}.transacciones")
        print(f"  2024-01-15 11:01     pipeline-job-service            createTable          {CATALOG}.{SCH_RIESGO}.alertas")
        print(f"  2024-01-15 10:59     riesgo-bot@nequi.com.co         selectFromTable      {CATALOG}.{SCH_PAGOS}.transacciones")
        print(f"  2024-01-15 10:45     cumplimiento@nequi.com.co       getTable             {CATALOG}.{SCH_RIESGO}.v_transacciones")
    else:
        raise

---

## Resumen del modulo

En este modulo construiste el primer Data Product de Delfos:

| Que hiciste | Comando | Por que importa |
|---|---|---|
| Verificar estructura de dominios | `SHOW SCHEMAS` | Principio 1: propiedad por dominio |
| Cargar 2.500 transacciones | `saveAsTable` via Spark | Datos reales para demostrar conceptos |
| Perfilar el Data Product | `GROUP BY canal / ciudad / hora` | Validar que el dato es correcto antes de publicar |
| Registrar el contrato | `ALTER TABLE SET TBLPROPERTIES` | Principio 2: datos como producto |
| Documentar columnas | `ALTER COLUMN COMMENT` | Catalogo autodescubrible sin documentacion externa |
| Ver el log de Delta | `DESCRIBE HISTORY` | Trazabilidad regulatoria sin sistema adicional |
| Crear vista con RLS | `CREATE VIEW` + `is_member()` | Principio 4: gobernanza federada |
| Time Travel | `VERSION AS OF` | Auditoria SFC: reproducir estado historico |
| Descubrir Data Products | `information_schema.tables` | Principio 3: plataforma self-serve |
| Auditar accesos | `system.access.audit` | SARLAFT: quien accedio a que y cuando |

---

## Reflexion y discusion

**Reflexion**

El equipo de clientes quiere analizar comportamiento de pago por ciudad para personalizar la app.
Deberian solicitar acceso directo a `pagos.transacciones` o pedir al equipo de pagos que publique
una vista agregada sin datos individuales de usuario? Que principio de Data Mesh aplica?

**Discusion**

Con Data Mesh, el equipo de pagos es dueno de sus transacciones. Un analista de riesgo necesita
un campo nuevo en la tabla. Quien decide si se agrega? En cuanto tiempo? Como se negocia el
cambio sin romper los consumidores existentes del Data Product?

**Referencia:** [Data Mesh Principles — Zhamak Dehghani](https://martinfowler.com/articles/data-mesh-principles.html)